# 03. Literały w czasie kompilacji

## Cele

Po tym notebooku potrafisz:

* wyjaśnić, kiedy funkcja (i operator literałowy) `constexpr` **musi** zostać obliczona podczas kompilacji,
  a kiedy może działać w czasie wykonania;
* przewidzieć, co się stanie, gdy funkcja `constexpr` dojdzie do `throw` w czasie kompilacji, a co — w czasie wykonania;
* użyć `consteval`, aby literał z błędną wartością był błędem kompilacji;
* napisać literał napisowy `operator""_x(const char*, std::size_t)` i przetestować jego logikę w GoogleTest.

## `constexpr`: kompilacja możliwa, ale nie zawsze gwarantowana

Funkcja `constexpr` **może** zostać obliczona podczas kompilacji — jeśli jej argumenty są znane. Standard
gwarantuje to jednak tylko tam, gdzie **wymagane jest wyrażenie stałe**: w inicjalizacji zmiennej `constexpr`,
w `static_assert`, w argumencie szablonu czy rozmiarze tablicy
([cppreference: Manifestly constant-evaluated expressions](https://en.cppreference.com/cpp/language/constant_expression#Manifestly_constant-evaluated_expressions)).
W każdym innym miejscu wywołanie zachowuje się jak wywołanie zwykłej funkcji — może zostać wykonane w czasie
działania programu
([C++ Core Guidelines F.4](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rf-constexpr):
„`constexpr` does not guarantee compile-time evaluation”,
[learncpp: Constexpr functions part 2](https://www.learncpp.com/cpp-tutorial/constexpr-functions-part-2/)).

Operator literałowy to zwykła funkcja, więc te same reguły dotyczą literałów `constexpr`. Przykład — literał
procentów z kontrolą zakresu 0–100:

```cpp
#include <stdexcept>

struct Percent {
    unsigned value;
};

constexpr Percent make_percent(unsigned long long v) {
    if (v > 100) {
        throw std::invalid_argument("percent out of range");
    }
    return Percent{static_cast<unsigned>(v)};
}

constexpr Percent operator""_pct(unsigned long long v) { return make_percent(v); }

constexpr Percent discount = 15_pct;  // constexpr variable: evaluated by the compiler
static_assert(discount.value == 15);  // static_assert: evaluated by the compiler
static_assert((100_pct).value == 100);

int main() {
    Percent tax = 23_pct;  // ordinary variable: may be evaluated at run time
    return tax.value == 23 ? 0 : 1;
}
```

## `throw` w funkcji `constexpr`

Funkcja `constexpr` może zawierać `throw` — o skutku decyduje to, **kiedy** do niego dojdzie
([cppreference: constexpr — Notes](https://en.cppreference.com/cpp/language/constexpr#Notes)):

* **podczas obliczania wyrażenia stałego** — `throw` sprawia, że wyrażenie nie jest stałe, więc w C++20
  to **błąd kompilacji** ([N4861 expr.const/5.24](https://timsong-cpp.github.io/cppwp/n4861/expr.const#5.24));
* **w czasie wykonania** — to zwykły wyjątek, który można złapać.

```cpp
#include <stdexcept>

struct Percent {
    unsigned value;
};

constexpr Percent make_percent(unsigned long long v) {
    if (v > 100) {
        throw std::invalid_argument("percent out of range");
    }
    return Percent{static_cast<unsigned>(v)};
}

constexpr Percent operator""_pct(unsigned long long v) { return make_percent(v); }

constexpr Percent bad = 150_pct;  // error: the throw is reached during constant evaluation
```

Ta sama wartość w zwykłej zmiennej **kompiluje się**, a błąd wychodzi dopiero w czasie działania programu —
test w GoogleTest łapie go przez `EXPECT_THROW`
([Krzemieński, *User-defined literals — Part I*](https://akrzemi1.wordpress.com/2012/08/12/user-defined-literals-part-i/):
literały nie są liczone w czasie kompilacji, „unless you initialize another compile-time constant”):

```cpp
#include <stdexcept>

#include <gtest/gtest.h>

struct Percent {
    unsigned value;
};

constexpr Percent make_percent(unsigned long long v) {
    if (v > 100) {
        throw std::invalid_argument("percent out of range");
    }
    return Percent{static_cast<unsigned>(v)};
}

constexpr Percent operator""_pct(unsigned long long v) { return make_percent(v); }

TEST(PercentLiteral, InvalidValueThrowsAtRunTime) {
    EXPECT_THROW(150_pct, std::invalid_argument);  // compiles: not a constant expression context
}
```

Uwaga: `const` to nie `constexpr`. `const Percent p = 150_pct;` też skompiluje się i rzuci wyjątek w czasie
wykonania ([C++ Core Guidelines Con.5](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rconst-constexpr)).

## `consteval`: zawsze podczas kompilacji

C++20 dodał słowo `consteval`. Funkcja `consteval` (*immediate function*) **musi** zostać obliczona podczas
kompilacji przy każdym wywołaniu — jeśli się nie da, program się nie kompiluje
([cppreference: consteval](https://en.cppreference.com/cpp/language/consteval#Explanation),
[P1073R3](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2018/p1073r3.html)).
Literał `consteval` z błędną wartością jest więc błędem kompilacji **wszędzie**, także w zwykłej zmiennej.

Sprawdzona recepta na literał z walidacją składa się z dwóch funkcji:

1. **funkcja `constexpr`** z logiką i `throw` (np. `make_percent`) — działa też w czasie wykonania, więc da się
   ją przetestować przez `EXPECT_THROW` i użyć z danymi spoza kodu źródłowego;
2. **operator literałowy `consteval`**, który tylko ją wywołuje — błędny literał w kodzie zatrzymuje kompilację.

```cpp
#include <stdexcept>

#include <gtest/gtest.h>

struct Percent {
    unsigned value;
};

constexpr Percent make_percent(unsigned long long v) {
    if (v > 100) {
        throw std::invalid_argument("percent out of range");
    }
    return Percent{static_cast<unsigned>(v)};
}

consteval Percent operator""_pct(unsigned long long v) { return make_percent(v); }

TEST(PercentLiteral, CreatesValidValue) {
    Percent tax = 23_pct;  // always evaluated by the compiler
    EXPECT_EQ(tax.value, 23u);
}

// 150_pct does not compile, so the error path is tested through make_percent
TEST(MakePercent, RejectsValuesAbove100) {
    EXPECT_THROW(make_percent(150), std::invalid_argument);
}
```

**Starsze kompilatory.** Clang 14 ma błąd: literał `consteval` użyty wewnątrz **szablonu** (także w
`TYPED_TEST`, w lambdzie z parametrem `auto`, w funkcji z parametrem `auto` czy w argumencie domyślnym) kończy się
błędem linkera `undefined reference to operator""…`. Poprawiono to w Clang 15
([LLVM #54578](https://github.com/llvm/llvm-project/issues/54578),
[Clang 15 Release Notes](https://releases.llvm.org/15.0.0/tools/clang/docs/ReleaseNotes.html#c-20-feature-support)).
Dlatego w tym laboratorium literałów `consteval` używamy tylko w zwykłych funkcjach i zwykłych `TEST`.

## Demo: błąd kompilacji na żądanie

Projekt [`demos/`](demos/CMakeLists.txt) zawiera literał `consteval` z powyższej recepty
([demos/percent.h](demos/percent.h)) i dwa cele:

* `percent_tests` ([demos/percent_test.cpp](demos/percent_test.cpp)) — testy poprawnych literałów i funkcji
  `make_percent`; budują się i przechodzą,
* `percent_compile_error` ([demos/compile_error.cpp](demos/compile_error.cpp)) — program z `150_pct`, który
  **celowo się nie kompiluje**; nie należy do domyślnego budowania, więc budujemy go osobno.

In [ ]:
# Configure, build and run the demo tests
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug --output-on-failure

Oczekiwany wynik: `100% tests passed` (2 testy).

In [ ]:
# Build the program that uses 150_pct (fails to compile on purpose)
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --target percent_compile_error

Kompilacja kończy się błędem. GCC wskazuje i wywołanie, i miejsce `throw`:

```
error: call to consteval function 'demo::literals::operator""_pct(150)' is not a constant expression
...
error: expression '<throw-expression>' is not a constant expression
```

Clang zgłasza `call to consteval function 'demo::literals::operator""_pct' is not a constant expression`.
Zmień `150` w [demos/compile_error.cpp](demos/compile_error.cpp) na wartość z zakresu 0–100, zapisz plik
i uruchom komórkę ponownie — tym razem program się zbuduje.

## Literały napisowe: `(const char*, std::size_t)`

Dla literału napisowego z sufiksem, np. `"release"_tag`, kompilator wywołuje `operator""_tag("release", 7)`:
pierwszy parametr wskazuje na znaki, drugi to **długość napisu bez kończącego `'\0'`**
([cppreference: User-defined literals — Syntax](https://en.cppreference.com/cpp/language/user_literal#Syntax),
[standard: lex.ext](https://eel.is/c++draft/lex.ext#5)). Z tej pary najwygodniej od razu zrobić
`std::string_view{s, n}` — tak samo działa standardowy `operator""sv`
([cppreference](https://en.cppreference.com/cpp/string/basic_string_view/operator%22%22sv)). Konstruktor
`string_view` jest `constexpr`, więc cała obróbka napisu może odbyć się podczas kompilacji.

Od C++14 funkcja `constexpr` może mieć pętle (`for`, także zakresowy, `while`) i zmieniać swoje zmienne lokalne
([N3652](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2013/n3652.html),
[cppreference: constexpr function](https://en.cppreference.com/cpp/language/constexpr#constexpr_function)),
a `string_view` daje dostęp do znaków przez `s[i]`:

```cpp
#include <cstddef>
#include <string_view>

constexpr std::size_t count_char(std::string_view s, char wanted) {
    std::size_t count = 0;  // a local variable that changes
    for (std::size_t i = 0; i < s.size(); ++i) {
        if (s[i] == wanted) {
            ++count;
        }
    }
    return count;
}

static_assert(count_char("1.2.3", '.') == 2);  // the whole loop runs in the compiler

int main() {}
```

```cpp
#include <cstddef>
#include <stdexcept>
#include <string_view>

struct Tag {
    std::string_view name;
};

constexpr Tag make_tag(std::string_view s) {
    if (s.empty() || s.size() > 16) {
        throw std::invalid_argument("a tag has 1..16 characters");
    }
    return Tag{s};
}

consteval Tag operator""_tag(const char* s, std::size_t n) { return make_tag(std::string_view{s, n}); }

constexpr std::string_view operator""_view(const char* s, std::size_t n) { return std::string_view{s, n}; }

static_assert("release"_tag.name == "release");
static_assert("a\0b"_view.size() == 3);  // n counts the embedded '\0', not the terminating one

int main() {
    Tag stable = "stable"_tag;
    // Tag none = ""_tag;  // error: call to consteval function ... is not a constant expression
    return stable.name.size() == 6 ? 0 : 1;
}
```

## Ćwiczenie 4 — kolory

*Treść ćwiczenia — w przygotowaniu.*

## Ćwiczenie 5 — czas trwania

*Treść ćwiczenia — w przygotowaniu.*

## Ciekawostki dla dociekliwych

### Forma surowa i szablon `template <char...>`

Literał liczbowy można też przekazać do operatora **jako tekst**. Forma surowa `operator""_x(const char*)`
dostaje znaki literału dokładnie tak, jak je zapisano — razem z przedrostkiem `0x` i separatorami.
Szablon `template <char... Cs> operator""_x()` dostaje te same znaki jako argumenty szablonu, więc
przed C++20 był jedynym sposobem, by wymusić przetwarzanie literału podczas kompilacji
([standard: over.literal](https://eel.is/c++draft/over.literal#5),
[Krzemieński, *User-defined literals — Part II*](https://akrzemi1.wordpress.com/2012/10/23/user-defined-literals-part-ii/)).
Obie formy są używane tylko wtedy, gdy brakuje formy `unsigned long long` / `long double`.

```cpp
#include <cstddef>
#include <cstring>
#include <iostream>

std::size_t operator""_chars(const char* s) { return std::strlen(s); }

template <char... Cs>
constexpr std::size_t operator""_count() { return sizeof...(Cs); }

int main() {
    std::cout << 0x1F_chars << ' ' << 1'000_chars << ' ' << 2.50_chars << '\n';  // 4 5 4
    static_assert(0x1F_count == 4);
}
```

### C++20: napis jako argument szablonu

Od C++20 parametrem szablonu może być obiekt klasy, np. „napis o stałej długości” (`fixed_string`). Literał
napisowy może wtedy trafić do operatora jako **argument szablonu**, a każdy napis daje osobną instancję
([P0732R2](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2018/p0732r2.pdf),
[cppreference: Literal operators](https://en.cppreference.com/cpp/language/user_literal#Literal_operators)).

```cpp
#include <cstddef>

template <std::size_t N>
struct FixedString {
    char data[N]{};
    constexpr FixedString(const char (&s)[N]) {
        for (std::size_t i = 0; i < N; ++i) {
            data[i] = s[i];
        }
    }
};

template <FixedString S>
constexpr std::size_t operator""_len() { return sizeof(S.data) - 1; }

static_assert("hello"_len == 5);

int main() {}
```

### Czy w ogóle używać własnych literałów?

Zdania są podzielone. Google C++ Style Guide zakazuje ich wprost — także tych z biblioteki standardowej —
bo tworzą składnię nieznaną nawet doświadczonym programistom
([Google C++ Style Guide: Operator Overloading](https://google.github.io/styleguide/cppguide.html#Operator_Overloading)).
C++ Core Guidelines nie mają reguły o definiowaniu literałów, ale same zalecają sufiksy standardowe:
`"Hello!"s`, `100ms`
([NL.11](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rl-literals),
[SL.str.12](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rstr-s)).

### C++26 łagodzi regułę o `throw`

W tym laboratorium obowiązuje C++20: `throw` osiągnięty podczas obliczania wyrażenia stałego to błąd.
Szkic C++26 pozwala na wyjątek, który zostanie **złapany** w tym samym obliczeniu; wyjątek niezłapany — jak
w `150_pct` — nadal jest błędem
([cppreference: Core constant expression](https://en.cppreference.com/cpp/language/constant_expression#Core_constant_expression)).

## Linki i materiały

* [cppreference: constexpr](https://en.cppreference.com/cpp/language/constexpr#constexpr_function) i
  [consteval](https://en.cppreference.com/cpp/language/consteval#Explanation)
* [learncpp: Constexpr functions part 3 and consteval](https://www.learncpp.com/cpp-tutorial/constexpr-functions-part-3-and-consteval/)
* [Microsoft Learn: constexpr](https://learn.microsoft.com/en-us/cpp/cpp/constexpr-cpp#constexpr_functions)
  — po polsku: [constexpr (C++)](https://learn.microsoft.com/pl-pl/cpp/cpp/constexpr-cpp#constexpr_functions) (tłumaczenie maszynowe)
* [N4861](https://timsong-cpp.github.io/cppwp/n4861/expr.const#13) — tekst standardu C++20 (wyrażenia stałe, funkcje natychmiastowe)
* [Jason Turner: Your New Mental Model of constexpr (CppCon 2021, wideo)](https://www.youtube.com/watch?v=MdrfPSUtMVM)
* [Rainer Grimm: Back to Basics: const and constexpr (CppCon 2021, wideo)](https://www.youtube.com/watch?v=tA6LbPyYdco)

---

[← poprzedni: 02. Literały definiowane przez użytkownika](../02_udl/02_udl.ipynb) | [Strona główna](../_start.ipynb)